# Multi-Model Demo — Bi-Mamba × Hybrid × Transformer (zh ↔ vi)

Notebook **demo linh hoạt** chạy bất kỳ trong 3 model trong cùng một file:

| `MODEL_KIND`   | Module                                | Config                               | Run dir                       |
|----------------|---------------------------------------|--------------------------------------|-------------------------------|
| `mamba`        | `bi_mamba_mt.model.BiMambaTranslator` | `configs/bi_mamba_55m.yaml`          | `runs/bi_mamba_55m/`          |
| `hybrid`       | `hybrid_mt.model.HybridMambaAttentionTranslator` | `configs/hybrid_mamba_attention.yaml` | `runs/hybrid_mamba_attention/` |
| `transformer`  | `transformer_mt.model.TransformerTranslator` | `configs/transformer_30m.yaml`       | `runs/transformer_30m/`       |

Cells dưới đây dùng dispatch theo `MODEL_KIND` ở **Cell 4** — đổi 1 dòng là đổi model.

Dùng để:
* Demo dịch sample list cho từng model (Cell 7).
* So sánh BLEU/chrF 3 model trên cùng test set (Cell 8) — yêu cầu đã có ckpt cho mỗi model.
* So sánh decoding sweep 3 model (Cell 9).

**Notebook KHÔNG train** — chỉ load ckpt + eval/translate. Train trong các notebook riêng:
`bi_mamba_zh_vi_colab.ipynb`, `transformer_zh_vi_colab.ipynb`, `hybrid_mamba_zh_vi_colab.ipynb`.

## 1. Clone repo

In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
WORK_DIR = '/content/NLP_DHM'

if not os.path.isdir(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
%cd {WORK_DIR}
!git pull --ff-only

## 2. Cài đặt dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

import sys
if 'src' not in sys.path:
    sys.path.insert(0, 'src')
print('sys.path[0] =', sys.path[0])


### 2b. (Tuỳ chọn) Mamba CUDA fast-path

Cần cho `mamba` và `hybrid` (encoder Bi-Mamba). Skip nếu chỉ chạy `transformer`.

In [ ]:
import torch
if torch.cuda.is_available():
    !pip install -q causal_conv1d mamba_ssm
    try:
        from causal_conv1d import causal_conv1d_fn
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
        print('Mamba CUDA fast-path available.')
    except Exception as e:
        print('Mamba CUDA fast-path NOT available (will fallback to PyTorch ref):', e)

## 3. **Chọn model** ⬇️

Đổi `MODEL_KIND` để switch model. Tất cả cells sau đều tuân theo lựa chọn này.

In [ ]:
# === Pick one ===
MODEL_KIND = 'hybrid'   # 'mamba' | 'hybrid' | 'transformer'

# Tuỳ chọn: override config / output_dir / checkpoint cụ thể.
# Để None thì dispatch dùng default ở dictionary _MODEL_REGISTRY phía dưới.
CONFIG_OVERRIDE = None   # e.g. 'configs/_colab.yaml'
CKPT_OVERRIDE = None     # e.g. 'runs/hybrid_mamba_attention/best_ema.pt'

## 4. Dispatch table

Chỉ duy nhất chỗ này biết về 3 model. Tất cả cells sau dùng `BUILD_MODEL` để load.

In [ ]:
import os, yaml, torch
from pathlib import Path

_MODEL_REGISTRY = {
    'mamba': dict(
        config_path = 'configs/bi_mamba_55m.yaml',
        run_dir     = 'runs/bi_mamba_55m',
        eval_script = 'scripts/evaluate.py',
        train_script = 'scripts/train.py',
    ),
    'hybrid': dict(
        config_path = 'configs/hybrid_mamba_attention.yaml',
        run_dir     = 'runs/hybrid_mamba_attention',
        eval_script = 'scripts/evaluate_hybrid.py',
        train_script = 'scripts/train_hybrid.py',
    ),
    'transformer': dict(
        config_path = 'configs/transformer_30m.yaml',
        run_dir     = 'runs/transformer_30m',
        eval_script = 'scripts/evaluate_transformer.py',
        train_script = 'scripts/train_transformer.py',
    ),
}

assert MODEL_KIND in _MODEL_REGISTRY, f'Unknown MODEL_KIND: {MODEL_KIND}'
META = dict(_MODEL_REGISTRY[MODEL_KIND])
META['config_path'] = CONFIG_OVERRIDE or META['config_path']

with open(META['config_path']) as f:
    CFG = yaml.safe_load(f)

print(f'MODEL_KIND   = {MODEL_KIND}')
print(f'config_path  = {META["config_path"]}')
print(f'run_dir      = {META["run_dir"]}')
print(f'eval_script  = {META["eval_script"]}')
print(f'train_script = {META["train_script"]}')


def BUILD_MODEL(model_cfg_dict):
    """Construct an empty model from `model_cfg_dict` and return it."""
    if MODEL_KIND == 'mamba':
        from bi_mamba_mt.model import BiMambaTranslator, ModelConfig
        return BiMambaTranslator(ModelConfig(**model_cfg_dict))
    if MODEL_KIND == 'hybrid':
        from hybrid_mt.model import HybridMambaAttentionTranslator, ModelConfig
        return HybridMambaAttentionTranslator(ModelConfig(**model_cfg_dict))
    if MODEL_KIND == 'transformer':
        from transformer_mt.model import TransformerTranslator, ModelConfig
        return TransformerTranslator(ModelConfig(**model_cfg_dict))
    raise ValueError(MODEL_KIND)


def PICK_CHECKPOINT(run_dir=None):
    """Pick the strongest available checkpoint in priority order."""
    if CKPT_OVERRIDE:
        return CKPT_OVERRIDE
    run_dir = run_dir or META['run_dir']
    for cand in ['avg_last5_ema.pt', 'best_ema.pt', 'avg_last5.pt', 'best.pt', 'latest_ema.pt', 'latest.pt']:
        p = os.path.join(run_dir, cand)
        if os.path.exists(p):
            return p
    return None

## 5. Load checkpoint cho `MODEL_KIND` đang chọn

In [ ]:
ckpt_path = PICK_CHECKPOINT()
if ckpt_path is None:
    print(f'⚠️  Không tìm thấy checkpoint trong {META["run_dir"]}/')
    print(f'    → train trước qua: python {META["train_script"]} --config {META["config_path"]}')
else:
    print(f'Loading {ckpt_path}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if ckpt_path is not None:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = BUILD_MODEL(ckpt.get('model_cfg', CFG['model'])).to(device).eval()
    model.load_state_dict(ckpt['model'], strict=False)
    if ckpt.get('is_ema'):
        print('  (EMA weights)')
    if ckpt.get('averaged_n'):
        print(f'  (averaged from {ckpt["averaged_n"]} checkpoints)')
    print(f'Params: {sum(p.numel() for p in model.parameters()):,}')
else:
    model = None

## 6. Demo dịch (sample list)

In [ ]:
from mt_base.tokenizer import Tokenizer
from mt_base.translate import translate_batch

if model is None:
    raise SystemExit('Cần checkpoint trước (cell 5).')

tokenizer = Tokenizer('data/tokenizer/spm.model')

zh_samples = [
    '你好，世界。',
    '今天天气真好。',
    '这本书我已经读完了。',
    '请问最近的地铁站在哪里？',
    '我明天下午三点要开会。',
]
vi_samples = [
    'Xin chào thế giới.',
    'Hôm nay trời thật đẹp.',
    'Tôi đã đọc xong cuốn sách này.',
    'Làm ơn cho tôi hỏi trạm tàu điện ngầm gần nhất ở đâu?',
    'Ngày mai 3 giờ chiều tôi có cuộc họp.',
]

# Length penalty defaults from config (per-direction).
lp_cfg = CFG['eval'].get('length_penalty', 1.0)
lp_zh2vi = lp_cfg.get('zh2vi', 1.0) if isinstance(lp_cfg, dict) else lp_cfg
lp_vi2zh = lp_cfg.get('vi2zh', 1.0) if isinstance(lp_cfg, dict) else lp_cfg
beam = int(CFG['eval'].get('beam_size', 4))

print(f'[{MODEL_KIND}] beam={beam} lp_zh2vi={lp_zh2vi} lp_vi2zh={lp_vi2zh}')
print('--- zh → vi ---')
for s, t in zip(zh_samples, translate_batch(model, tokenizer, zh_samples, 'zh2vi', beam_size=beam, length_penalty=lp_zh2vi)):
    print(f'  {s!r}\n  -> {t!r}')

print('--- vi → zh ---')
for s, t in zip(vi_samples, translate_batch(model, tokenizer, vi_samples, 'vi2zh', beam_size=beam, length_penalty=lp_vi2zh)):
    print(f'  {s!r}\n  -> {t!r}')

## 7. Eval BLEU/chrF (chiều + length-buckets) cho model đang chọn

In [ ]:
if ckpt_path is None:
    raise SystemExit('Cần checkpoint trước (cell 5).')

!python {META['eval_script']} \
    --config {META['config_path']} \
    --checkpoint {ckpt_path} \
    --num-samples 5000 \
    --beam-size 4 \
    --length-buckets

## 8. So sánh 3 model trên cùng test set

Eval lần lượt cả 3 model (nếu ckpt tồn tại). Dùng cùng `--num-samples`, cùng `--beam-size`, cùng test set.

In [ ]:
import os, subprocess, re, collections

NUM_SAMPLES = 5000
BEAM = 4

results = collections.defaultdict(dict)
for kind, info in _MODEL_REGISTRY.items():
    ck = None
    for cand in ['avg_last5_ema.pt', 'best_ema.pt', 'avg_last5.pt', 'best.pt']:
        p = os.path.join(info['run_dir'], cand)
        if os.path.exists(p):
            ck = p
            break
    if ck is None:
        print(f'[{kind}] no checkpoint in {info["run_dir"]}/  → skip')
        continue
    print(f'\n=== {kind} :: {ck} ===')
    cmd = [
        'python', info['eval_script'],
        '--config', info['config_path'],
        '--checkpoint', ck,
        '--num-samples', str(NUM_SAMPLES),
        '--beam-size', str(BEAM),
    ]
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print('STDERR:', p.stderr[-500:])
        continue
    for d in ('zh2vi', 'vi2zh'):
        m = re.search(rf'\[{d}\]\s+n=(\d+)\s+BLEU=([\d.]+)\s+chrF=([\d.]+)', p.stdout)
        if m:
            results[kind][d] = dict(n=int(m.group(1)), bleu=float(m.group(2)), chrf=float(m.group(3)))

print('\n\n=== Summary ===')
print(f'{"model":<12} {"zh2vi BLEU":>12} {"chrF":>8}   {"vi2zh BLEU":>12} {"chrF":>8}')
for kind in ['mamba', 'hybrid', 'transformer']:
    if kind in results:
        z = results[kind].get('zh2vi', {})
        v = results[kind].get('vi2zh', {})
        print(f'{kind:<12} {z.get("bleu", 0):>12.2f} {z.get("chrf", 0):>8.2f}   {v.get("bleu", 0):>12.2f} {v.get("chrf", 0):>8.2f}')

## 9. Decode sweep cho model đang chọn (CSV)

Dùng `scripts/sweep_decode.py --model-kind {MODEL_KIND}`. Chạy mất ~30-60 phút trên T4 cho beam ∈ {1,2,4,6}.

In [ ]:
import os
if ckpt_path is None:
    raise SystemExit('Cần checkpoint trước (cell 5).')

csv_path = os.path.join(META['run_dir'], 'sweep.csv')
!python scripts/sweep_decode.py \
    --model-kind {MODEL_KIND} \
    --config {META['config_path']} \
    --checkpoint {ckpt_path} \
    --num-samples 2000 \
    --beams 1 2 4 6 \
    --lp-zh2vi 0.8 0.9 1.0 1.1 1.2 \
    --lp-vi2zh 0.6 0.8 0.9 1.0 \
    --length-buckets \
    --out {csv_path}

## 10. ML Interpretation

Khi đã có cả 3 model trong bảng Cell 8:

| Quan sát                                  | Diễn giải                              | Hành động      |
|-------------------------------------------|----------------------------------------|----------------|
| Hybrid ≫ Bi-Mamba và ≈ Transformer        | Mamba decoder là bottleneck            | Giữ Hybrid    |
| Hybrid ≈ Bi-Mamba và ≪ Transformer        | Bi-Mamba encoder yếu cho MT            | Replace encoder |
| Cả 3 ≈ nhau (~20 BLEU)                   | Bottleneck là data/tokenizer           | Data ablation |
| Hybrid > Transformer (rare)               | Hybrid là kiến trúc tối ưu             | Production    |

Cũng nên xem **bucket breakdown** (Cell 7) — nếu một model rớt mạnh ở bucket `long` (≥50 chars), đó là dấu hiệu cụt câu / EOS bias / decoder context limit, không phải lỗi data.